# Optimal transport with Sinkhorn

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/sinkhorn_transport)

How do you move a pile of sand shaped like a ring into two heaps, and then into a spiral, with the least total effort? Optimal transport finds the plan $P_{ij}$ that moves mass from $x_i$ to $y_j$ at cost $|x_i - y_j|^2$; its minimum is the squared Wasserstein distance $W_2^2$. Adding a little entropy $\varepsilon$ makes the problem solvable by Sinkhorn's alternating updates, written here in the log domain with logsumexp for a whole batch of $\varepsilon$ at once; as $\varepsilon \to 0$ the cost drops to the exact optimum found by the Hungarian algorithm.

**Vector form**

$$W_2^2(\mu, \nu) = \min_{P\mathbf 1 = a,\ P^\top\mathbf 1 = b,\ P \ge 0} \langle P, C\rangle,\qquad \mathrm{OT}_\varepsilon = \min_P \langle P, C\rangle + \varepsilon\,\mathrm{KL}(P \,\|\, a \otimes b),\qquad P = \operatorname{diag}(a\,e^{f/\varepsilon})\,e^{-C/\varepsilon}\operatorname{diag}(b\,e^{g/\varepsilon})$$

**Index (tensor) form**

$$C_{ij} = (x_{ik} - y_{jk})(x_{ik} - y_{jk}),\qquad f_{ei} = -\varepsilon_e \log \sum_j b_j\,e^{(g_{ej} - C_{ij})/\varepsilon_e},\qquad g_{ej} = -\varepsilon_e \log \sum_i a_i\,e^{(f_{ei} - C_{ij})/\varepsilon_e},\qquad P_{eij} = a_i b_j\,e^{(f_{ei} + g_{ej} - C_{ij})/\varepsilon_e}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ik},\ y_{jk}$ | `ring(N), two_blobs(N), spiral(N)` | (N, 2) | synthetic point clouds generated from formulas and random numbers, each point with mass $a_i = 1/N$ |
| $C_{ij}$ | `einsum('ijk,ijk->ij', x[:, None] - y[None], x[:, None] - y[None])` | (N, M) | squared distance, repeated $k$ ⇒ summed |
| $\varepsilon_e$ | `torch.logspace(0, log10(eps_min), E)` | (E,) | a batch of regularisation strengths solved at the same time |
| $f_{ei}$ | `-eps * logsumexp(log_b + (g[:, None] - C) / eps, dim=-1)` | (E, N) | Sinkhorn update in the log domain, sum over $j$ as a logsumexp; never overflows even for tiny $\varepsilon$ |
| $g_{ej}$ | `-eps * logsumexp(log_a[:, None] + (f[..., None] - C) / eps, dim=-2)` | (E, M) | the same over $i$: after it, the column sums of $P$ equal $b$ exactly |
| $P_{eij}$ | `exp(log_a[:, None] + log_b + (f[..., None] + g[:, None] - C) / eps)` | (E, N, M) | transport plan: blurred for large $\varepsilon$, almost a permutation for small $\varepsilon$ |
| $T(x_i)$ | `einsum('ij,jk->ik', P, y) / P.sum(-1, keepdim=True)` | (N, 2) | barycentric map: where the mass of $x_i$ goes; points move on straight lines $(1-t)\,x + t\,T(x)$ (McCann interpolation) |
| $S_\varepsilon$ | `sinkhorn_divergence(x, a, y, b, eps, n_iter)` | (E,) | $\mathrm{OT}_\varepsilon(\mu,\nu) - \tfrac12\mathrm{OT}_\varepsilon(\mu,\mu) - \tfrac12\mathrm{OT}_\varepsilon(\nu,\nu)$, zero when $\mu = \nu$ |

Monge posed the problem in 1781 and Kantorovich relaxed it to plans $P$ in 1942. Cuturi (2013) showed that the entropic version is solved by Sinkhorn's matrix scaling, a few matrix products per iteration, which made optimal transport practical in machine learning. Small $\varepsilon$ needs care: the code works with the potentials $f, g$ and logsumexp instead of $e^{-C/\varepsilon}$ (which underflows), and anneals $\varepsilon$ from 1 down to its target ($\varepsilon$-scaling). The cost $\langle P_\varepsilon, C\rangle$ decreases to the exact $W_2^2$ computed by the Hungarian algorithm (a slow exact reference, checked against brute force over all permutations), while the dual value $\mathrm{OT}_\varepsilon$ carries an entropic bias that the Sinkhorn divergence of Feydy et al. (2019) removes. The tests also check two closed forms: in one dimension the optimal plan pairs sorted points (quantile functions), and between Gaussians $W_2^2 = |m_1 - m_2|^2 + \operatorname{tr}(S_1 + S_2 - 2(S_1^{1/2} S_2 S_1^{1/2})^{1/2})$ (the Bures metric), using Brenier's theorem that the gradient of a convex function is the optimal map. All shapes are generated by the code; no images are used.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/sinkhorn_transport/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import barycentric_map, cost_matrix, gaussian_w2_squared, w2_squared_1d

CHAIN = ["ring", "two blobs", "spiral", "ring"]


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    n = 60 if fast else params["n_points"]
    n_iter = 100 if fast else params["n_iter"]
    eps = tensors.epsilons(params["eps_min"], 1.0, 4 if fast else params["n_eps"], dev)  # (E,) from 1 down
    a = tensors.uniform_weights(n, dev)
    clouds = [tensors.SHAPES[name](n, k, dev) for k, name in enumerate(CHAIN[:-1])]
    clouds.append(clouds[0])

    legs = []
    for k in range(len(CHAIN) - 1):
        x, y = clouds[k], clouds[k + 1]
        c = cost_matrix(x, y)  # (N, N)
        out = solve.sinkhorn(a, a, c, eps, n_iter, eps_start=1.0)
        exact = c[torch.arange(n), solve.exact_assignment(c)].mean().item()  # Hungarian: the ε → 0 limit
        big = eps[eps >= 0.03]
        div = solve.sinkhorn_divergence(x, a, y, a, big, n_iter // 3)
        marg = (out["plan"].sum(-1) - a).abs().max() / a[0]
        name = f"{CHAIN[k]} → {CHAIN[k + 1]}"
        print(f"{name}: N = {n} points each, exact W2² (Hungarian) = {exact:.4f}")
        for e, cost, dual in zip(eps.tolist(), out["cost"].tolist(), out["dual"].tolist()):
            print(f"  ε = {e:7.4f}: <P, C> = {cost:.4f}   OT_ε = {dual:.4f}")
        print(
            f"  Sinkhorn divergence at ε = {big[-1]:.3f}: S_ε = {div[-1]:.4f} "
            f"(entropic bias removed; OT_ε there {out['dual'][len(big) - 1]:.4f})"
        )
        print(f"  largest relative marginal error {marg:.1e}; plan ≥ 0: {bool((out['plan'] >= 0).all())}")
        tx = barycentric_map(out["plan"][-1], y)  # where each source point goes at the smallest ε
        legs.append(
            {
                "x": x.cpu(),
                "y": y.cpu(),
                "tx": tx.cpu(),
                "plan": out["plan"].cpu(),
                "cost": out["cost"].cpu(),
                "dual": out["dual"].cpu(),
                "exact": exact,
                "div": (big.cpu(), div.cpu()),
                "name": name,
            }
        )

    x1, y1 = clouds[0][:, 0], clouds[1][:, 0]  # 1D check: the x-coordinates of ring and blobs
    c1 = cost_matrix(x1[:, None], y1[:, None])
    one_d = solve.sinkhorn(a, a, c1, eps[-1:], n_iter, eps_start=1.0)["cost"][0]
    print(
        f"1D (x-coordinates only): quantile formula W2² = {w2_squared_1d(x1, y1):.5f}, "
        f"Sinkhorn at ε = {eps[-1]:.4f}: {one_d:.5f}"
    )
    m1, m2 = torch.zeros(2, dtype=torch.float64), torch.tensor([1.0, 0.0], dtype=torch.float64)
    s1, s2 = torch.eye(2, dtype=torch.float64), torch.diag(torch.tensor([0.25, 2.0], dtype=torch.float64))
    print(
        "Gaussians N(0, I) → N((1, 0), diag(1/4, 2)): Bures closed form "
        f"W2² = {gaussian_w2_squared(m1, s1, m2, s2):.4f}"
    )
    return plot.animate(legs, eps.cpu(), out_dir / "sinkhorn.gif", 6 if fast else 22)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/sinkhorn_transport/equation.py
r"""Optimal transport → tensor translation.

Two discrete measures μ = Σ_i a_i δ_{x_i} and ν = Σ_j b_j δ_{y_j} and the cost C_ij = |x_i − y_j|². A coupling P ≥ 0
with row sums a and column sums b says how much mass moves from x_i to y_j; the squared Wasserstein-2 distance is

    W_2^2(\mu, \nu) = \min_{P\,\mathbf 1 = a,\ P^\top \mathbf 1 = b} \sum_{ij} P_{ij} C_{ij}.

Entropic OT (Cuturi 2013) adds ε KL(P | a ⊗ b). The optimum has the form P_ij = a_i b_j e^{(f_i + g_j − C_ij)/ε},
and Sinkhorn's alternating updates of the potentials f, g (here in the log domain, with logsumexp) converge to it:

    f_i = -\varepsilon \log \sum_j b_j\,e^{(g_j - C_{ij})/\varepsilon},\qquad
    g_j = -\varepsilon \log \sum_i a_i\,e^{(f_i - C_{ij})/\varepsilon}.

As ε → 0 the plan sharpens to the exact (unregularised) optimum. The Sinkhorn divergence
S_ε(μ, ν) = OT_ε(μ, ν) − ½ OT_ε(μ, μ) − ½ OT_ε(ν, ν) removes the entropic bias, so S_ε(μ, μ) = 0.

Closed forms used as checks: in 1D the optimal plan pairs sorted points (quantile functions); between Gaussians
W_2² = |m_1 − m_2|² + tr(S_1 + S_2 − 2 (S_1^{1/2} S_2 S_1^{1/2})^{1/2}) (the Bures metric).
"""

import torch


def cost_matrix(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """C_ij = |x_i − y_j|² = (x_ik − y_jk)(x_ik − y_jk)   (k summed).   (N, d), (M, d) → (N, M)"""
    diff = x[:, None, :] - y[None, :, :]
    return torch.einsum("ijk,ijk->ij", diff, diff)


def update_f(g: torch.Tensor, c: torch.Tensor, log_b: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""f_i = −ε logsumexp_j(log b_j + (g_j − C_ij)/ε)   (j summed).   (E, M), (N, M), (M,), (E, 1, 1) → (E, N)"""
    return -eps[:, :, 0] * torch.logsumexp(log_b + (g[:, None, :] - c) / eps, dim=-1)


def update_g(f: torch.Tensor, c: torch.Tensor, log_a: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""g_j = −ε logsumexp_i(log a_i + (f_i − C_ij)/ε)   (i summed).   (E, N), (N, M), (N,), (E, 1, 1) → (E, M)"""
    return -eps[:, :, 0] * torch.logsumexp(log_a[:, None] + (f[:, :, None] - c) / eps, dim=-2)


def plan(f: torch.Tensor, g: torch.Tensor, c: torch.Tensor, log_a, log_b, eps: torch.Tensor) -> torch.Tensor:
    """P_ij = a_i b_j exp((f_i + g_j − C_ij)/ε).   (E, N), (E, M), (N, M) → (E, N, M)"""
    return torch.exp(log_a[:, None] + log_b + (f[:, :, None] + g[:, None, :] - c) / eps)


def barycentric_map(p: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """T(x_i) = Σ_j P_ij y_j / Σ_j P_ij: where the mass of x_i goes on average.   (N, M), (M, d) → (N, d)"""
    return torch.einsum("ij,jk->ik", p, y) / p.sum(-1, keepdim=True)


def interpolate(x: torch.Tensor, tx: torch.Tensor, t: float) -> torch.Tensor:
    """McCann's displacement interpolation: every particle moves on a straight line, x_t = (1 − t) x + t T(x)."""
    return (1 - t) * x + t * tx


def w2_squared_1d(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """1D closed form for equal-size uniform samples: mean (x_(i) − y_(i))² of the sorted values.   (N,), (N,) → ()"""
    return (torch.sort(x).values - torch.sort(y).values).pow(2).mean()


def sqrtm(s: torch.Tensor) -> torch.Tensor:
    """Symmetric positive square root through the eigendecomposition S = U diag(λ) Uᵀ.   (d, d) → (d, d)"""
    lam, u = torch.linalg.eigh(s)
    return u @ torch.diag(lam.clamp(min=0).sqrt()) @ u.T


def gaussian_w2_squared(m1, s1, m2, s2) -> torch.Tensor:
    """W_2² between N(m1, S1) and N(m2, S2): |m1 − m2|² + tr(S1 + S2 − 2 (S1^{½} S2 S1^{½})^{½}).   → ()"""
    r1 = sqrtm(s1)
    return (m1 - m2).pow(2).sum() + torch.trace(s1 + s2 - 2 * sqrtm(r1 @ s2 @ r1))


def gaussian_monge_matrix(s1: torch.Tensor, s2: torch.Tensor) -> torch.Tensor:
    """A = S1^{−½} (S1^{½} S2 S1^{½})^{½} S1^{−½}: the optimal map between centred Gaussians is x ↦ A x.   (d, d)"""
    r1 = sqrtm(s1)
    r1_inv = torch.linalg.inv(r1)
    return r1_inv @ sqrtm(r1 @ s2 @ r1) @ r1_inv


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/sinkhorn_transport/tensors.py
"""Build the concrete tensors: synthetic point clouds (ring, two blobs, spiral), weights and Gaussian samples.

Every shape is generated here from a formula and random numbers, no images are used.
"""

import math

import torch


def ring(n: int, seed: int, device: torch.device) -> torch.Tensor:
    """n points on a noisy circle of radius 1: (n, 2)."""
    gen = torch.Generator().manual_seed(seed)
    t = 2 * math.pi * torch.rand(n, generator=gen, dtype=torch.float64)
    r = 1 + 0.06 * torch.randn(n, generator=gen, dtype=torch.float64)
    return torch.stack([r * torch.cos(t), r * torch.sin(t)], -1).to(device)


def two_blobs(n: int, seed: int, device: torch.device) -> torch.Tensor:
    """n points in two Gaussian blobs at (±0.9, 0.2), the left one wider: (n, 2)."""
    gen = torch.Generator().manual_seed(seed)
    side = (torch.arange(n) % 2).double() * 2 - 1  # −1, +1, −1, …
    z = torch.randn(n, 2, generator=gen, dtype=torch.float64)
    scale = torch.where(side < 0, 0.32, 0.22)[:, None]
    return (z * scale + torch.stack([0.9 * side, torch.full_like(side, 0.2)], -1)).to(device)


def spiral(n: int, seed: int, device: torch.device) -> torch.Tensor:
    """n points along an Archimedean spiral r = 0.12 + 0.95 s, angle 3π s, s ∈ [0, 1]: (n, 2)."""
    gen = torch.Generator().manual_seed(seed)
    s = torch.rand(n, generator=gen, dtype=torch.float64).sqrt()  # more points on the long outer arms
    r, t = 0.12 + 0.95 * s, 3 * math.pi * s
    pts = torch.stack([r * torch.cos(t), r * torch.sin(t)], -1)
    return (pts + 0.03 * torch.randn(n, 2, generator=gen, dtype=torch.float64)).to(device)


SHAPES = {"ring": ring, "two blobs": two_blobs, "spiral": spiral}


def uniform_weights(n: int, device: torch.device) -> torch.Tensor:
    """a_i = 1/n: (n,) float64."""
    return torch.full((n,), 1.0 / n, dtype=torch.float64, device=device)


def epsilons(eps_min: float, eps_max: float, count: int, device: torch.device) -> torch.Tensor:
    """count values of ε, log-spaced from eps_max down to eps_min: (E,)."""
    return torch.logspace(math.log10(eps_max), math.log10(eps_min), count, dtype=torch.float64, device=device)


def gaussian_samples(m: torch.Tensor, s: torch.Tensor, n: int, device: torch.device, seed: int) -> torch.Tensor:
    """n samples of N(m, S) as m + L z with S = L Lᵀ (Cholesky): (n, d)."""
    gen = torch.Generator().manual_seed(seed)
    z = torch.randn(n, len(m), generator=gen, dtype=torch.float64).to(device)
    return m + z @ torch.linalg.cholesky(s).T


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/sinkhorn_transport/solve.py
"""Sinkhorn in the log domain for a batch of ε, the Sinkhorn divergence, and an exact reference (Hungarian)."""

import numpy as np
import torch

from problems._common.progress import steps

from .equation import cost_matrix, plan, update_f, update_g


def sinkhorn(
    a: torch.Tensor, b: torch.Tensor, c: torch.Tensor, eps: torch.Tensor, n_iter: int, eps_start: float | None = None
) -> dict:
    """Entropic OT for every ε_e at once. With eps_start, ε is annealed geometrically from eps_start to its target
    over the first half of the iterations (ε-scaling, warm-started potentials), which makes small ε converge.
    a (N,), b (M,), C (N, M), ε (E,) → plan (E, N, M), potentials f (E, N), g (E, M), cost ⟨P, C⟩ (E,),
    dual ⟨f, a⟩ + ⟨g, b⟩ = OT_ε (E,)."""
    log_a, log_b = a.log(), b.log()
    target = eps[:, None, None].to(c.dtype)  # (E, 1, 1)
    start = target.clamp(min=eps_start) if eps_start is not None else target
    g = torch.zeros(len(eps), c.shape[1], dtype=c.dtype, device=c.device)
    for t in steps(n_iter, "Sinkhorn iterations"):
        frac = min(1.0, 2 * t / n_iter) if eps_start is not None else 1.0
        e = start * (target / start) ** frac  # (E, 1, 1) current ε
        f = update_f(g, c, log_b, e)
        g = update_g(f, c, log_a, e)
    p = plan(f, g, c, log_a, log_b, target)
    return {
        "plan": p,
        "f": f,
        "g": g,
        "cost": torch.einsum("eij,ij->e", p, c),
        "dual": f @ a + g @ b,
    }


def sinkhorn_divergence(x, a, y, b, eps: torch.Tensor, n_iter: int) -> torch.Tensor:
    """S_ε = OT_ε(μ, ν) − ½ OT_ε(μ, μ) − ½ OT_ε(ν, ν) (Feydy et al. 2019): zero when μ = ν.   → (E,)"""
    xy = sinkhorn(a, b, cost_matrix(x, y), eps, n_iter)["dual"]
    xx = sinkhorn(a, a, cost_matrix(x, x), eps, n_iter)["dual"]
    yy = sinkhorn(b, b, cost_matrix(y, y), eps, n_iter)["dual"]
    return xy - 0.5 * xx - 0.5 * yy


def exact_assignment(c: torch.Tensor) -> torch.Tensor:
    """Exact optimal assignment (Hungarian algorithm with potentials, O(n³)): the slow, exact reference for ε → 0.
    C (N, M) with N ≤ M → column assigned to every row (N,)."""
    cost = c.detach().cpu().double().numpy()
    n, m = cost.shape
    u, v = np.zeros(n + 1), np.zeros(m + 1)
    p = np.zeros(m + 1, dtype=int)  # p[j] = row assigned to column j (1-based, 0 = free)
    way = np.zeros(m + 1, dtype=int)
    for i in range(1, n + 1):
        p[0], j0 = i, 0
        minv = np.full(m + 1, np.inf)
        used = np.zeros(m + 1, dtype=bool)
        while True:
            used[j0] = True
            i0 = p[j0]
            free = ~used[1:]
            cur = cost[i0 - 1] - u[i0] - v[1:]  # reduced costs from row i0
            better = free & (cur < minv[1:])
            minv[1:][better] = cur[better]
            way[1:][better] = j0
            j1 = int(np.flatnonzero(free)[np.argmin(minv[1:][free])]) + 1
            delta = minv[j1]
            u[p[used]] += delta
            v[used] -= delta
            minv[1:][free] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while j0:  # augment along the alternating path
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
    rows = np.zeros(n, dtype=int)
    for j in range(1, m + 1):
        if p[j]:
            rows[p[j] - 1] = j - 1
    return torch.from_numpy(rows).to(c.device)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/sinkhorn_transport/plot.py
"""Points flowing from one shape to the next along the plan, the plan for a sweep of ε, and the cost against ε."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection

from problems._common.plotting import frame_indices, save_gif

from .equation import interpolate

LINES = 70  # source points whose plan rows are drawn
TOP = 6  # largest plan entries drawn per row


def animate(legs: list[dict], eps: torch.Tensor, path: Path, per_leg: int) -> Path:
    """legs: one dict per transport (x, y, tx (N, 2), plan (E, N, N), cost, dual (E,), exact, div, name)."""
    hold = max(2, per_leg // 5)
    frames = [(k, min(1.0, s / per_leg)) for k in range(len(legs)) for s in range(per_leg + hold)]
    e = eps.numpy()
    rows = np.linspace(0, len(legs[0]["x"]) - 1, LINES).astype(int)

    fig = plt.figure(figsize=(11, 3.9))
    grid = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.2], wspace=0.3, bottom=0.15)
    ax_f = fig.add_subplot(grid[0])
    ax_p = fig.add_subplot(grid[1])
    for ax in (ax_f, ax_p):
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    empty = np.zeros((0, 2))
    target = ax_f.scatter(empty[:, 0], empty[:, 1], s=6, color="#d1d5db")
    moving = ax_f.scatter(empty[:, 0], empty[:, 1], s=7, color="#2563eb")
    title_f = ax_f.set_title("", fontsize=10)
    src_p = ax_p.scatter(empty[:, 0], empty[:, 1], s=4, color="#93c5fd")
    dst_p = ax_p.scatter(empty[:, 0], empty[:, 1], s=4, color="#fca5a5")
    lines = LineCollection([], linewidths=0.6)
    ax_p.add_collection(lines)
    title_p = ax_p.set_title("", fontsize=10)

    ax_c = fig.add_subplot(grid[2])
    (l_cost,) = ax_c.semilogx([], [], "o-", color="#2563eb", ms=3, label="⟨P_ε, C⟩")
    (l_dual,) = ax_c.semilogx([], [], "s--", color="#9333ea", ms=3, label="OT_ε (dual value)")
    (l_div,) = ax_c.semilogx([], [], "^-", color="#16a34a", ms=3, label="Sinkhorn divergence S_ε")
    l_exact = ax_c.axhline(0, color="#dc2626", ls=":", lw=1.3, label="exact W₂² (Hungarian)")
    marker = ax_c.axvline(e[0], color="#f59e0b", lw=1)
    ax_c.set_xlim(e.max() * 1.4, e.min() * 0.7)  # ε decreases to the right
    ax_c.set_xlabel("regularisation ε → 0")
    ax_c.set_ylabel("transport cost")
    ax_c.legend(fontsize=7, frameon=False, loc="upper left")
    title_c = ax_c.set_title("", fontsize=9)

    def update(i: int):
        k, t = frames[i]
        leg = legs[k]
        x, y = leg["x"].numpy(), leg["y"].numpy()
        target.set_offsets(y)
        moving.set_offsets(interpolate(leg["x"], leg["tx"], t).numpy())
        title_f.set_text(f"{leg['name']}, t = {t:.2f}")
        j = int(round(t * (len(e) - 1)))  # ε sharpens while the points travel
        p = leg["plan"][j].numpy()
        src_p.set_offsets(x)
        dst_p.set_offsets(y)
        top = np.argsort(p[rows], axis=1)[:, -TOP:]  # (LINES, TOP) columns of the largest entries
        segs = np.stack([np.repeat(x[rows], TOP, axis=0), y[top.ravel()]], axis=1)  # (LINES·TOP, 2, 2)
        weight = np.take_along_axis(p[rows], top, axis=1) / p[rows].sum(1, keepdims=True)
        lines.set_segments(segs)
        lines.set_color([(0.22, 0.25, 0.32, min(1.0, w)) for w in weight.ravel()])
        title_p.set_text(f"plan P_ε (largest entries), ε = {e[j]:.3g}")
        cost, dual = leg["cost"].numpy(), leg["dual"].numpy()
        big, div = (v.numpy() for v in leg["div"])
        l_cost.set_data(e, cost)
        l_dual.set_data(e, dual)
        l_div.set_data(big, div)
        l_exact.set_ydata([leg["exact"], leg["exact"]])
        ax_c.set_ylim(min(0.0, div.min()) - 0.02, max(cost.max(), dual.max()) * 1.08)
        marker.set_xdata([e[j], e[j]])
        title_c.set_text(f"{leg['name']}: W₂² = {leg['exact']:.3f}, ⟨P,C⟩ at ε = {e[-1]:.3g} is {cost[-1]:.3f}")

    return save_gif(fig, update, frame_indices(len(frames), 90), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_points': 250,
    'eps_min': 0.002,
    'n_eps': 8,
    'n_iter': 600
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.sinkhorn_transport")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.sinkhorn_transport.main")
problem_dir = Path("problems/maths/sinkhorn_transport")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
